# 01 · Theozyme & contig (Stages 1–2)

Build a catalytic-constraint (`.cst`) theozyme and a validated
RFdiffusion contig, and exercise the guard-rails the protocol calls out.

In [1]:
import sys, os
# make the package importable from the notebooks/ directory
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
EXAMPLES = os.path.join(ROOT, "examples")
print("project root:", ROOT)


project root: /home/user/biofx_python/enzyme_design


## Build a theozyme from catalytic geometry

In [2]:
from enzyme_design.theozyme import (Theozyme, ConstraintBlock, AtomMap,
                                     GeometricConstraint)
block = ConstraintBlock(
    res1=AtomMap(1, ['OG','CB','CA'], ['SER']),
    res2=AtomMap(2, ['C1','O1','O2'], ['LIG']),
    constraints=[
        GeometricConstraint('distanceAB', 2.80, 0.20, 100.0),
        GeometricConstraint('angle_A', 105.0, 5.0, 50.0, 360.0),
    ],
    comment='Ser-OG nucleophilic attack on substrate carbonyl')
theo = Theozyme([block], title='demo')
print(theo.to_cst())

# demo
# --- constraint 1 ---
CST::BEGIN
  # Ser-OG nucleophilic attack on substrate carbonyl
  TEMPLATE::   ATOM_MAP: 1 atom_name: OG CB CA ,
  TEMPLATE::   ATOM_MAP: 1 residue3: SER
  TEMPLATE::   ATOM_MAP: 2 atom_name: C1 O1 O2 ,
  TEMPLATE::   ATOM_MAP: 2 residue3: LIG
  CONSTRAINT:: distanceAB:     2.80   0.20   100.0    0.0 1
  CONSTRAINT:: angle_A:   105.00   5.00    50.0  360.0 1
CST::END



## Round-trip: parse the `.cst` back
Serialise → parse must preserve atoms, residue identities, constraints.

In [3]:
from enzyme_design.theozyme import parse_cst
parsed = parse_cst(theo.to_cst())
assert parsed.blocks[0].res1.atoms == ['OG','CB','CA']
assert parsed.catalytic_residues() == ['SER']
print('round-trip OK; catalytic residues =', parsed.catalytic_residues())

round-trip OK; catalytic residues = ['SER']


## Build & validate a contig

In [4]:
from enzyme_design.contig import build_contig, parse_contig
c = build_contig([('A', 84, 87)], flank=(10, 120), total_length=(150, 150))
print('contigs:', c.to_hydra()[0])
print('length :', c.to_hydra()[1])
print('residue range:', c.length_range(), '  valid:', c.validate() == [])

contigs: contigmap.contigs=['10-120,A84-87,10-120']
length : contigmap.length='150-150'
residue range: (24, 244)   valid: True


## Guard-rail 1 — bare integers are rejected
The protocol warns: *always give ranges (e.g. `16-16`), never bare
integers.* The parser turns that mistake into a hard error.

In [5]:
from enzyme_design.contig import ContigError
try:
    parse_contig("['120,A84-87,10-120']")
except ContigError as e:
    print('correctly rejected:', e)

correctly rejected: bare integer '120' in contig: generated spans must be a range (write 120-120, not 120). See protocol Stage 2, step 6.


## Guard-rail 2 — impossible total length is caught
`contigmap.length` must be reachable from the segment ranges.

In [6]:
bad = parse_contig("['10-20,A84-87,10-20']", '150-150')
print('problems:', bad.validate())

problems: ['contigmap.length=150-150 is impossible: segments allow 24-44 residues']


## Multi-island motif (typical real active site)

In [7]:
multi = build_contig([('A', 84, 85), ('A', 120, 121)],
                     flank=(10, 50), inter_island=(5, 30),
                     total_length=(80, 120))
print(multi.to_contigs_string())
print('motif residues:', multi.motif_residues())

10-50,A84-85,5-30,A120-121,10-50
motif residues: ['A84', 'A85', 'A120', 'A121']
